# PHASE 5: Deep Learning with PyTorch
**Traceability**
- Issue ID: #5 Deep Learning Implementation

## 1. Objectives
- Implement a sequence-based deep learning solution using PyTorch.
- Transform time-series data into 3D sequences using a sliding window approach.
- Design and train an LSTM (Long Short-Term Memory) network for RUL prediction.
- Perform hyperparameter tuning and evaluate model performance.

### 5.1 Import Libraries & Configure PyTorch
We import PyTorch and set up the device (CPU/GPU) and random seeds.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility Config ──────────────────────────────────────────────
np.random.seed(42)
torch.manual_seed(42)

# ── Global Config ────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')
SEQ_LEN = 50  # Number of past cycles to consider
BATCH_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

COLORS = ['#1F4E79', '#2E75B6', '#70AD47', '#FF7043', '#AB47BC']

### 5.2 Sequence Data Preparation
Deep learning models for time-series require data in 3D format: `(samples, time_steps, features)`. We use a sliding window to generate these sequences.

In [ ]:
def create_sequences(df, seq_len, sensor_cols):
    """Transform dataframe into 3D sequences for LSTM."""
    sequences = []
    targets = []
    
    for unit in df['unit_number'].unique():
        unit_df = df[df['unit_number'] == unit].sort_values('time_cycles')
        data = unit_df[sensor_cols].values
        target = unit_df['RUL'].values
        
        for i in range(len(data) - seq_len + 1):
            sequences.append(data[i:i+seq_len])
            targets.append(target[i+seq_len-1])
            
    return np.array(sequences), np.array(targets)

class CMAPSSDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = torch.FloatTensor(sequences)
        self.targets = torch.FloatTensor(targets)
        
    def __len__(self):
        return len(self.targets)
        
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

### 5.3 Model Architecture: LSTM
We define a multi-layer LSTM network to capture temporal dependencies, followed by fully connected layers for regression.

In [ ]:
class RULPredictorLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim=1):
        super(RULPredictorLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, output_dim)
        )
        
    def forward(self, x):
        # x shape: (batch, seq_len, input_dim)
        out, _ = self.lstm(x)
        # out shape: (batch, seq_len, hidden_dim)
        # We only need the output of the last time step
        out = out[:, -1, :]
        out = self.fc(out)
        return out.squeeze()

### 5.4 Training Pipeline
Define the training loop with early stopping and loss tracking.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=50, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        model.train()
        batch_losses = []
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())
            
        avg_train_loss = np.mean(batch_losses)
        train_losses.append(avg_train_loss)
        
        model.eval()
        val_batch_losses = []
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(DEVICE), y_val.to(DEVICE)
                val_outputs = model(x_val)
                val_loss = criterion(val_outputs, y_val)
                val_batch_losses.append(val_loss.item())
        
        avg_val_loss = np.mean(val_batch_losses)
        val_losses.append(avg_val_loss)
        
        if (epoch+1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
            
    return train_losses, val_losses

### 5.5 Execution & Evaluation
Load data, scale features, prepare sequences, train the model, and visualize results.

In [ ]:
# 1. Load Data
df_train = pd.read_csv(PROCESSED_DIR / 'train_labeled.csv')
df_test = pd.read_csv(PROCESSED_DIR / 'test_labeled.csv')
sensor_cols = [c for c in df_train.columns if c.startswith('s_')]

# 2. Scaling
scaler = StandardScaler()
df_train[sensor_cols] = scaler.fit_transform(df_train[sensor_cols])
df_test[sensor_cols] = scaler.transform(df_test[sensor_cols])

# 3. Create Sequences
X_train, y_train = create_sequences(df_train, SEQ_LEN, sensor_cols)
X_test, y_test = create_sequences(df_test, SEQ_LEN, sensor_cols)

# 4. DataLoaders
train_size = int(0.8 * len(X_train))
train_ds = CMAPSSDataset(X_train[:train_size], y_train[:train_size])
val_ds = CMAPSSDataset(X_train[train_size:], y_train[train_size:])
test_ds = CMAPSSDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

# 5. Initialize Model
model = RULPredictorLSTM(input_dim=len(sensor_cols), hidden_dim=64, num_layers=2).to(DEVICE)

# 6. Train
train_losses, val_losses = train_model(model, train_loader, val_loader, epochs=30)

# 7. Plot Loss Curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', color=COLORS[0])
plt.plot(val_losses, label='Val Loss', color=COLORS[3])
plt.title('Training and Validation Loss (MSE)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# 8. Save Model
torch.save(model.state_dict(), '../artifacts/pytorch_lstm.pth')
print("✅ PyTorch model trained and saved.")